In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import os

In [2]:
p1 = [0.2,0.1,0.05,0.03,0.02,0.02,0.02,0.02,0.02,0.03,
      0.05,0.08,0.2,0.45,0.6,0.75,0.65,0.7,0.55,0.45,
      0.5,0.45,0.4,0.35,0.4,0.35,0.3,0.25,0.22,0.2,
      0.25,0.3,0.35,0.4,0.45,0.48,0.42,0.45,0.35,0.3,
      0.25,0.3,0.35,0.25,0.2,0.15,0.1,0.15]

p2 = [0.1,0.09,0.075,0.06,0.05,0.04,0.03,0.025,0.03,0.04,
      0.06,0.15,0.3,0.45,0.6,0.925,0.65,0.725,0.55,0.475,
      0.45,0.475,0.425,0.4,0.35,0.3,0.275,0.25,0.225,0.225,
      0.25,0.275,0.325,0.4,0.525,0.55,0.625,0.55,0.5,0.4,
      0.35,0.3,0.275,0.25,0.3,0.225,0.15,0.125]

p3 = [0.1,0.083333,0.066667,0.1,0.116667,0.1,0.083333,0.083333,0.066667,0.1,
      0.116667,0.166667,0.266667,0.366667,0.5,0.4,0.333333,0.383333,0.283333,0.25,
      0.233333,0.216667,0.2,0.183333,0.2,0.183333,0.166667,0.16,0.166667,0.173333,
      0.166667,0.183333,0.233333,0.333333,0.5,0.433333,0.466667,0.4,0.333333,0.266667,
      0.2,0.166667,0.25,0.216667,0.233333,0.25,0.216667,0.133333]

p4 = [0.05,0.025,0.02,0.0125,0.0125,0.0125,0.0125,0.0125,0.025,0.05,
      0.075,0.2,0.375,0.625,0.8,0.875,0.525,0.475,0.5,0.45,
      0.375,0.325,0.275,0.25,0.225,0.2125,0.225,0.2,0.1875,0.175,
      0.1875,0.2125,0.25,0.325,0.425,0.5,0.65,0.55,0.625,0.55,
      0.45,0.375,0.3,0.25,0.2,0.15,0.1,0.075]

times = []
current = datetime(2026, 1, 1, 0, 30)

for i in range(17520):
    times.append(current.strftime("%H:%M:%S"))
    current += timedelta(minutes=30)

print(times[:5])
print(times[-5:])

['00:30:00', '01:00:00', '01:30:00', '02:00:00', '02:30:00']
['22:00:00', '22:30:00', '23:00:00', '23:30:00', '00:00:00']


In [3]:
def get_probs(people):
    if people < 1.5:
        return p1
    elif people < 2.5:
        return p2
    elif people < 3.5:
        return p3
    else:
        return p4

def simulate_day(num_people, probs, weekend=False):
    showers_done = [0] * num_people
    medium_done = [0] * num_people

    MEDIUM_LOAD = 10 if weekend else 7.5

    preferences = [
        "morning" if random.random() < 0.7 else "evening"
        for _ in range(num_people)
    ]

    ltot = []

    for count in range(48):
        step_total = 0

        for person in range(num_people):
            r = random.random()
            load = 0

            if not weekend:
                morning_range = range(11, 17)
            else:
                morning_range = range(15, 21)

            evening_range = range(34, 39)

            if count in morning_range:
                if (preferences[person] == "morning" and
                    not showers_done[person] and r < probs[count]):
                    load = 35
                    showers_done[person] = 1
                elif r < probs[count] * 0.3:
                    load = 0.5

            elif 17 <= count <= 33:
                if r < probs[count] * 0.2:
                    load = 0.5

            elif count in evening_range:
                if (preferences[person] == "evening" and
                    not showers_done[person] and r < probs[count]):
                    load = 35
                    showers_done[person] = 1
                elif (not medium_done[person] and r < probs[count]):
                    load = MEDIUM_LOAD
                    medium_done[person] = 1
                elif r < probs[count] * 0.3:
                    load = 0.5

            step_total += load

        ltot.append(step_total)

    return np.array(ltot)

def simulate_year(num_people, probs):

    start_date = datetime(2026, 1, 1)
    end_date = datetime(2026, 12, 31)

    current_date = start_date
    ltot_year = []

    while current_date <= end_date:

        # -----------------------
        # WEEKEND CHECK
        # -----------------------
        weekend = current_date.weekday() >= 5  # 5=Sat, 6=Sun

        # -----------------------
        # SEASONAL MULTIPLIER
        # -----------------------
        if ((current_date.month == 12 and current_date.day >= 23) or
            (current_date.month == 1 and current_date.day <= 2)):
            multiplier = 1.2
        else:
            multiplier = 1.0

        # -----------------------
        # SIMULATE DAY
        # -----------------------
        day_litres = simulate_day(num_people, probs, weekend)

        # Apply multiplier SAFELY
        day_litres = np.array(day_litres) * multiplier

        # Store results
        ltot_year.extend(day_litres.tolist())

        # Move to next day
        current_date += timedelta(days=1)

    return np.array(ltot_year)

TARGET_LEN = 17520  # 365 days × 48 half-hours

def pad_series(x, target=TARGET_LEN, fill_value=0.0):
    x = np.array(x, dtype=float)

    if len(x) >= target:
        return x[:target]

    padded = np.full(target, fill_value, dtype=float)
    padded[:len(x)] = x
    return padded

def solar_power(G, Ta):
    eta_0 = 0.668
    a1 = 1.5
    a2 = 0.0065

    Tm = (60 + Ta) / 2

    if G <= 0:
        return 0

    deltaT = Tm - Ta

    efficiency = eta_0 - (a1 * deltaT / G) - (a2 * deltaT**2 / G)
    efficiency = max(0, efficiency)

    return 1.78 * G * efficiency


def tilted_irradiance_full(DNI, DHI, day_of_year, hour,
                           lat=56.05, tilt=45, longitude=-4.22):

    L_std = 0
    time_correction = (L_std - longitude) / 15
    solar_time = hour + 0.5 + time_correction

    delta = 23.45 * np.sin(np.radians(360 * (284 + day_of_year) / 365))
    h = 15 * (solar_time - 12)

    cos_theta_z = (
        np.sin(np.radians(lat)) * np.sin(np.radians(delta)) +
        np.cos(np.radians(lat)) * np.cos(np.radians(delta)) * np.cos(np.radians(h))
    )
    cos_theta_z = max(0, cos_theta_z)

    cos_theta = (
        np.sin(np.radians(delta)) * np.sin(np.radians(lat - tilt)) +
        np.cos(np.radians(delta)) * np.cos(np.radians(lat - tilt)) * np.cos(np.radians(h))
    )
    cos_theta = max(0, cos_theta)

    G_beam = DNI * cos_theta
    G_diffuse = DHI * (1 + np.cos(np.radians(tilt))) / 2

    rho = 0.2
    GHI = DNI * cos_theta_z + DHI
    G_reflected = GHI * rho * (1 - np.cos(np.radians(tilt))) / 2

    return G_beam + G_diffuse + G_reflected


def simulate_tank_dynamic(
    ltot_year,
    people,
    eff_profile,
    ground_temp_profile,
    DNI,
    DHI,
    air_temp
):

    tank_volume = 135 if people <= 2 else 195
    #tank_volume = 200 if people <= 2 else 260
    
    if people < 3:
        heater_power = 8000
    else:
        heater_power = 13000

    timestep_hours = 0.5

    tank_temp_boiler = 60
    tank_temp_solar = 70  # NEW

    # Initial energy at boiler temp
    initial_energy = tank_volume * 4180 * (tank_temp_boiler - ground_temp_profile[0]) / 3600000

    tank_energy_solar = 0
    tank_energy_boiler = initial_energy

    boiler_profile = []
    solar_used_profile = []
    demand_profile = []
    solar_generated_profile = []

    for i in range(len(ltot_year)):

        litres_used = ltot_year[i]
        Tg = ground_temp_profile[i]
        Ta = air_temp[i]
        eff = eff_profile[i]

        # Time
        day_index = i // 48
        current_date = datetime(2026, 1, 1) + timedelta(days=day_index)
        day_of_year = current_date.timetuple().tm_yday
        hour = (i % 48) / 2

        # -----------------------
        # ENERGY LIMITS
        # -----------------------
        max_energy_boiler = tank_volume * 4180 * (tank_temp_boiler - Tg) / 3600000
        max_energy_solar  = tank_volume * 4180 * (tank_temp_solar  - Tg) / 3600000

        # -----------------------
        # ☀️ SOLAR GENERATION
        # -----------------------
        G = tilted_irradiance_full(DNI[i], DHI[i], day_of_year, hour)
        solar_p = solar_power(G, Ta) * 2
        solar_energy = solar_p * timestep_hours / 1000

        solar_generated_profile.append(solar_p)

        # -----------------------
        # ☀️ SOLAR CHARGING (UP TO 65°C)
        # -----------------------
        total_energy = tank_energy_solar + tank_energy_boiler

        if total_energy < max_energy_solar:
            tank_energy_solar += solar_energy

        # Clip ONLY to solar max
        total_energy = tank_energy_solar + tank_energy_boiler
        if total_energy > max_energy_solar:
            excess = total_energy - max_energy_solar
            tank_energy_solar = max(tank_energy_solar - excess, 0)

        # -----------------------
        # 🚿 DEMAND (still based on 60°C delivery)
        # -----------------------
        demand_energy = (4180 * (tank_temp_boiler - Tg) * litres_used) / 3600000

        solar_used = min(tank_energy_solar, demand_energy)
        tank_energy_solar -= solar_used

        remaining = demand_energy - solar_used

        boiler_used_from_tank = min(tank_energy_boiler, remaining)
        tank_energy_boiler -= boiler_used_from_tank

        remaining -= boiler_used_from_tank

        boiler_energy = 0
        if remaining > 0:
            max_boiler_energy_step = heater_power * timestep_hours / 1000
            boiler_energy = min(max_boiler_energy_step, remaining)

        # -----------------------
        # 🔥 LOSSES
        # -----------------------
        loss_per_hour = 0.08
        loss = loss_per_hour * timestep_hours

        total_energy = tank_energy_solar + tank_energy_boiler

        if total_energy > 0:
            solar_loss = loss * (tank_energy_solar / total_energy)
            boiler_loss = loss * (tank_energy_boiler / total_energy)
        else:
            solar_loss = boiler_loss = 0

        tank_energy_solar = max(tank_energy_solar - solar_loss, 0)
        tank_energy_boiler = max(tank_energy_boiler - boiler_loss, 0)

        # -----------------------
        # 🔥 BOILER CONTROL (ONLY TO 60°C)
        # -----------------------
        reheat_threshold = 0.8 * max_energy_boiler

        if (hour >= 22 or hour <= 5) and (tank_energy_solar + tank_energy_boiler) < reheat_threshold:
            energy_needed = max_energy_boiler - (tank_energy_solar + tank_energy_boiler)

            energy_added = min(heater_power * timestep_hours / 1000, energy_needed)

            tank_energy_boiler += energy_added
            boiler_energy += energy_added

        # -----------------------
        # OUTPUTS
        # -----------------------
        demand_power = demand_energy * 1000 / timestep_hours
        solar_used_power = solar_used * 1000 / timestep_hours

        thermal_boiler_power = boiler_energy * 1000 / timestep_hours
        boiler_power = thermal_boiler_power / eff if eff > 0 else 0

        demand_profile.append(demand_power)
        solar_used_profile.append(solar_used_power)
        boiler_profile.append(boiler_power)

    return (
        np.array(boiler_profile),
        np.array(solar_used_profile),
        np.array(demand_profile),
        np.array(solar_generated_profile)
    )

In [4]:
# %%
# ================================
# STEP 7: MAIN LOOP OVER BUILDINGS
# Description: Run full simulation for every building folder
# ================================
 
# Get all building folders
baseline_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"
building_ids = [f for f in os.listdir(baseline_path) if f.isdigit()]
 
 
print(f"Total buildings found: {len(building_ids)}")
print("Sample:", building_ids[:5])
 
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_lat_long_4_dec.csv"
ground_temp_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/half_hourly_ground_temp.csv"
direct_normal_solar_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/direct_normal_solar.csv"
diffuse_horizontal_solar_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/diffuse_horizontal_solar.csv"
air_temp_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/cleaned_OAtemp.csv"
 
# EPC data
epc_df = pd.read_csv(epc_path)
 
# Ground temperature data (half-hourly)
gt_df = pd.read_csv(ground_temp_path)
ground_temp_profile = gt_df["ground_temp_30mm"].values
 
# Direct normal solar
DNI_df = pd.read_csv(direct_normal_solar_path)
DNI_profile = DNI_df["direct_normal_solar"].values
 
# Diffuse horizontal solar
DHS_df = pd.read_csv(diffuse_horizontal_solar_path)
DHI_profile = DHS_df["diffuse_horizontal_solar"].values
 
# Air temperature (force numeric)
air_temp_df = pd.read_csv(air_temp_path)
air_temp_profile = pd.to_numeric(air_temp_df.iloc[:, 1], errors="coerce").values
 
 
#print(f"People: {people}")
print(f"Ground temp length: {len(ground_temp_profile)}")
print(f"Direct normal solar length: {len(DNI_profile)}")
print(f"Diffuse horizontal solar length: {len(DHI_profile)}")
print(f"Air temperature length: {len(air_temp_profile)}")
 
# solar inputs: missing data = no solar
DNI_profile = pad_series(DNI_profile, TARGET_LEN, 0.0)
DHI_profile = pad_series(DHI_profile, TARGET_LEN, 0.0)
# air temperature: fallback to mean climate
air_temp_profile = pad_series(air_temp_profile, TARGET_LEN, np.mean(air_temp_profile))
 
ground_temp_profile = pad_series(ground_temp_profile, TARGET_LEN, np.mean(ground_temp_profile))
 
for building_id in building_ids:
    try:
        print(f"\nProcessing building: {building_id}")
 
        # Get occupants
        people = int(
            epc_df.loc[
                epc_df["BUILDING_REFERENCE_NUMBER"] == int(building_id),
                "OCCUPANTS_ROUNDED_UP"
            ].values[0]
        )
 
        probs = get_probs(people)
 
        # Paths
        eff_path = f"{baseline_path}/{building_id}/HOT_WATER/water_heat_efficiency.csv"
        output_path = f"/Users/jakegehrung/Desktop/data_raw/baseline_models/{building_id}/HOT_WATER/solar_hot_water_load.csv"
 
        # Load efficiency
        eff_df = pd.read_csv(eff_path)
        eff_profile = eff_df["water_heat_efficiency"].values
 
        eff_profile = pad_series(eff_profile, TARGET_LEN, np.mean(eff_profile))
        # Simulate
        ltot_year = simulate_year(people, probs)
        boiler_profile, solar_used_profile, demand_profile, solar_generated_profile = simulate_tank_dynamic(ltot_year, 
                                                                                                            people, 
                                                                                                            eff_profile, 
                                                                                                            ground_temp_profile,
                                                                                                            DNI_profile,
                                                                                                            DHI_profile, 
                                                                                                            air_temp_profile)
 
        # ALWAYS align time vector to actual simulation length
        n = len(demand_profile)
        # regenerate correct time series (NO multiplication)
        times_full = times[:n]
        # Save
        df = pd.DataFrame({
            "Time": times_full,
            "hot_water_demand_W": demand_profile[:n],
            "solar_thermal_used_W": solar_used_profile[:n],
            "boiler_power_used_W": boiler_profile[:n],
            "solar_themal_generated_W": solar_generated_profile[:n]
        })
 
        df.to_csv(output_path, index=False)
 
        print(f"Saved: {output_path}")
 
    except Exception as e:
        print(f"Skipped {building_id} due to error: {e}")

Total buildings found: 117
Sample: ['1001664924', '1001829085', '1001063639', '1001829071', '1234761002']
Ground temp length: 17520
Direct normal solar length: 17519
Diffuse horizontal solar length: 17519
Air temperature length: 17520

Processing building: 1001664924
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/HOT_WATER/solar_hot_water_load.csv

Processing building: 1001829085
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/HOT_WATER/solar_hot_water_load.csv

Processing building: 1001063639
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063639/HOT_WATER/solar_hot_water_load.csv

Processing building: 1001829071
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829071/HOT_WATER/solar_hot_water_load.csv

Processing building: 1234761002
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761002/HOT_WATER/solar_hot_water_load.csv

Processing building: 1002539407
Saved: /Users/jakegehrung/Desktop/data_raw